# STEP 1: Anthropic API 最初のリクエスト

このノートブックでは、Claude API への基本的なリクエストを実践します。

## 1. 必要なパッケージのインストール

In [ ]:
%pip install anthropic python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 989.7 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [anthropic]14 [anthropic]

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /Users/kuniyasakai/Documents/workspace/claude_api_introduction/.venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. 環境変数の読み込みとクライアントの作成

In [2]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"

print("クライアントの作成に成功しました")

クライアントの作成に成功しました


## 3. 最初のAPIリクエスト

In [3]:
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "量子コンピューティングとは何ですか？一文で答えてください"
        }
    ]
)

print(message)

Message(id='msg_018qTe7st3QVtrfDN8UoUMAi', container=None, content=[TextBlock(citations=None, text='量子コンピューティングとは、量子力学の重ね合わせやもつれといった性質を利用して、従来のコンピュータでは困難な計算を高速に処理する技術です。', type='text')], model='claude-sonnet-4-5-20250929', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=30, output_tokens=65, server_tool_use=None, service_tier='standard'))


## 4. レスポンスからテキストを抽出

In [4]:
print(message.content[0].text)

量子コンピューティングとは、量子力学の重ね合わせやもつれといった性質を利用して、従来のコンピュータでは困難な計算を高速に処理する技術です。


---

# STEP 2: 複数ターンの会話（会話履歴の管理）

Claude はリクエスト間で会話を記憶しません。
文脈を維持するには、**会話履歴をこちら側で管理**してリクエストに含める必要があります。

## 5. ヘルパー関数の定義

In [5]:
def add_user_message(messages, text):
    """ユーザーメッセージを履歴に追加する"""
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    """アシスタントメッセージを履歴に追加する"""
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    """会話履歴全体をClaudeに送信してレスポンスを返す"""
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text

print("ヘルパー関数の定義が完了しました")

ヘルパー関数の定義が完了しました


## 6. 複数ターンの会話を実行する

In [6]:
# 空のメッセージリストからスタート
messages = []

# 最初の質問を追加
add_user_message(messages, "量子コンピューティングを一文で定義してください")

# Claudeの回答を取得
answer = chat(messages)
print(f"Claude (1回目): {answer}")

# Claudeの回答を履歴に追加
add_assistant_message(messages, answer)

# フォローアップの質問を追加
add_user_message(messages, "もう一文書いてください")

# 会話履歴全体を含めてリクエスト
final_answer = chat(messages)
print(f"Claude (2回目): {final_answer}")

Claude (1回目): 量子力学の重ね合わせやもつれなどの原理を利用して、従来のコンピュータでは困難な計算を高速に処理する次世代コンピューティング技術です。
Claude (2回目): 従来のビット（0か1）ではなく、0と1を同時に表現できる量子ビット（キュービット）を使うことで、複数の計算を並列的に実行できます。


## 7. 送信している会話履歴を確認する

In [7]:
# 2回目のリクエスト時点でのメッセージ履歴を確認
import json
print(json.dumps(messages, indent=2, ensure_ascii=False))

[
  {
    "role": "user",
    "content": "量子コンピューティングを一文で定義してください"
  },
  {
    "role": "assistant",
    "content": "量子力学の重ね合わせやもつれなどの原理を利用して、従来のコンピュータでは困難な計算を高速に処理する次世代コンピューティング技術です。"
  },
  {
    "role": "user",
    "content": "もう一文書いてください"
  }
]


---

# STEP 3: システムプロンプト

システムプロンプトを使うと、Claude の振る舞い・トーン・役割をカスタマイズできます。
ユーザーの質問とは別に、会話全体を通じて有効な「指示」として機能します。

## 8. chat関数をシステムプロンプト対応に更新する

Claude の API は `system=None` を受け付けないため、指定がある場合だけ含める実装にします。

In [8]:
def chat(messages, system=None):
    """会話履歴全体をClaudeに送信してレスポンスを返す（システムプロンプト対応）"""
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    # system=None の場合はパラメータに含めない
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

print("chat関数の更新が完了しました")

chat関数の更新が完了しました


## 9. システムプロンプトなしで質問する（通常の回答）

In [9]:
messages = []
add_user_message(messages, "5x + 2 = 3 はどうやって解きますか？")

answer = chat(messages)
print("【システムプロンプトなし】")
print(answer)

【システムプロンプトなし】
# 5x + 2 = 3 の解き方

この一次方程式を解く手順を説明します。

## 解法

**目標：xだけを左辺に残す**

### ステップ1：両辺から2を引く
```
5x + 2 = 3
5x + 2 - 2 = 3 - 2
5x = 1
```

### ステップ2：両辺を5で割る
```
5x = 1
5x ÷ 5 = 1 ÷ 5
x = 1/5
```

## 答え
**x = 1/5（または 0.2）**

## 検算
元の式に代入して確認：
```
5 × (1/5) + 2 = 1 + 2 = 3 ✓
```

正しく解けています！


## 10. システムプロンプトあり（数学家庭教師として振る舞う）

In [10]:
system_prompt = """
あなたは忍耐強い数学の家庭教師です。
生徒の質問に直接答えないでください。
ステップバイステップで解決策へ導いてください。
"""

messages = []
add_user_message(messages, "5x + 2 = 3 はどうやって解きますか？")

answer = chat(messages, system=system_prompt)
print("【システムプロンプトあり（数学家庭教師）】")
print(answer)

【システムプロンプトあり（数学家庭教師）】
いい質問ですね！一緒に考えていきましょう。

方程式 5x + 2 = 3 を解くには、xを孤立させる必要があります。

**まず最初の質問:**
xの項（5x）を孤立させるために、最初にどの項を取り除くべきだと思いますか？左辺には何がありますか？

ヒント：xと一緒にない数字を見てください。


---

# STEP 4: Temperature（温度）

`temperature` は Claude の回答の「ランダム性・創造性」を制御するパラメータです。

| 温度 | 特性 | 用途例 |
|------|------|--------|
| 0.0〜0.3（低） | 決定論的・一貫性が高い | コード生成、データ抽出、事実回答 |
| 0.4〜0.7（中） | バランス型 | 要約、教育コンテンツ、問題解決 |
| 0.8〜1.0（高） | 多様・創造的 | ブレインストーミング、創作、マーケティング |

## 11. chat関数にtemperatureパラメータを追加する

In [11]:
def chat(messages, system=None, temperature=1.0):
    """会話履歴全体をClaudeに送信してレスポンスを返す（temperature対応）"""
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

print("chat関数の更新が完了しました")

chat関数の更新が完了しました


## 12. 低温（0.0）vs 高温（1.0）を比較する

同じプロンプトを温度違いで3回ずつ実行して、回答のばらつきを確認します。

In [12]:
prompt = "タイムトラベルをテーマにした映画のアイデアを一文で教えてください。"

print("=== 低温 (temperature=0.0) ===")
for i in range(3):
    messages = []
    add_user_message(messages, prompt)
    answer = chat(messages, temperature=0.0)
    print(f"  {i+1}回目: {answer}")

print()
print("=== 高温 (temperature=1.0) ===")
for i in range(3):
    messages = []
    add_user_message(messages, prompt)
    answer = chat(messages, temperature=1.0)
    print(f"  {i+1}回目: {answer}")

=== 低温 (temperature=0.0) ===
  1回目: 「過去を変えるたびに別の平行世界が生まれ、主人公は無数の自分と出会いながら、最も幸せな世界線を探す旅に出る」
  2回目: 「過去を変えようとするたびに未来の自分が現れて阻止しにくる、終わりなき自己との戦いの物語」
  3回目: 「過去を変えるたびに別の誰かの記憶が消えていくことに気づいた科学者が、最後に自分自身の存在を賭けた選択を迫られる」

=== 高温 (temperature=1.0) ===
  1回目: 「過去の自分にメッセージを送れるアプリを手に入れた主人公が、些細な助言を繰り返すうちに現在が徐々に崩壊していくことに気づくサスペンス」
  2回目: 過去の自分に会いに行った主人公が、実は未来の自分も同じ瞬間に訪れていたことに気づき、複数の時間軸の自分たちが協力して世界の破滅を防ぐ物語。
  3回目: 「未来から送られてきた一枚の家族写真に写る見知らぬ人物が、実は過去を変えようとする自分自身だったことに気づく物語」


---

# STEP 5: ストリーミング

通常のリクエストは応答が完成するまで待つのに対し、ストリーミングはテキストが生成されるたびにチャンクごとに受信できます。
チャットアプリのような「文字が流れてくる」UXを実現するための機能です。

## 13. ストリームイベントの中身を確認する

`stream=True` を指定すると、どんなイベントが送られてくるかを見てみます。

In [13]:
messages = []
add_user_message(messages, "架空のデータベースを一文で説明してください")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_01VwW4NvwsiXjvDCfCueyp6D', container=None, content=[], model='claude-sonnet-4-5-20250929', role='assistant', stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=23, output_tokens=8, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='架空のデータベース', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='とは、実在しない想定上の組織や状況における', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='情報を管理するために設計された、教育・訓練・テ', type='text_delta'), index=0, type='content_block_del

## 14. テキストだけをリアルタイムで表示する

SDKの簡略インターフェース `messages.stream` を使うと、テキスト以外のイベントを自動でフィルタリングできます。
`end=""` により改行なしで文字が流れるように表示されます。

In [14]:
messages = []
add_user_message(messages, "架空のデータベースを一文で説明してください")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

架空のデータベースとは、実在しない想定上の組織やシステムのために作成された、教育・テスト・デモンストレーション目的で使用される仮想的なデータの集合体です。

## 15. ストリーミングしながら完全なメッセージも取得する

リアルタイム表示と同時に、DB保存や後続処理用の完全なメッセージオブジェクトも取得できます。

In [15]:
messages = []
add_user_message(messages, "架空のデータベースを一文で説明してください")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    # リアルタイムでテキストを表示
    for text in stream.text_stream:
        print(text, end="", flush=True)

    # ストリーミング完了後に完全なメッセージを取得
    final_message = stream.get_final_message()

print("\n\n--- final_message ---")
print(f"stop_reason : {final_message.stop_reason}")
print(f"input_tokens: {final_message.usage.input_tokens}")
print(f"output_tokens: {final_message.usage.output_tokens}")

架空のデータベースとは、実在しない組織や人物、出来事などの創作データを格納・管理するデータベースのことです。

--- final_message ---
stop_reason : end_turn
input_tokens: 23
output_tokens: 47


---

# STEP 6: 構造化データの出力制御（プリフィル + 停止シーケンス）

ClaudeはデフォルトでJSONをマークダウンコードブロックや説明文で囲んで返します。
**アシスタントメッセージのプリフィル**と**停止シーケンス**を組み合わせることで、生データだけを取得できます。

## 16. デフォルト（プリフィルなし）の出力を確認する

In [16]:
messages = []
add_user_message(messages, "EventBridgeルールをJSONで生成してください（短くてOK）")

text = chat(messages)
print(text)

```json
{
  "name": "MyEventRule",
  "description": "Example EventBridge rule",
  "eventPattern": {
    "source": ["aws.ec2"],
    "detail-type": ["EC2 Instance State-change Notification"],
    "detail": {
      "state": ["running"]
    }
  },
  "state": "ENABLED",
  "targets": [
    {
      "id": "1",
      "arn": "arn:aws:lambda:ap-northeast-1:123456789012:function:MyFunction"
    }
  ]
}
```

EC2インスタンスが起動状態になったときにLambda関数を実行するシンプルなルールです。


## 17. chat関数に stop_sequences パラメータを追加する

In [17]:
def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    """会話履歴全体をClaudeに送信してレスポンスを返す（stop_sequences対応）"""
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

print("chat関数の更新が完了しました")

chat関数の更新が完了しました


## 18. プリフィル + 停止シーケンスで生JSONだけを取得する

- `add_assistant_message(messages, "```json")` でClaudeに「コードブロックはもう始まっている」と思わせる
- `stop_sequences=["` ``` `"]` でClaudeが閉じタグを書こうとした瞬間に停止させる

In [18]:
import json

messages = []
add_user_message(messages, "EventBridgeルールをJSONで生成してください（短くてOK）")
add_assistant_message(messages, "```json")  # プリフィル：コードブロックが既に始まっているように見せる

text = chat(messages, stop_sequences=["```"])  # 閉じタグが来たら停止

print("--- 生のレスポンス ---")
print(repr(text))

print("\n--- パース済みJSON ---")
clean_json = json.loads(text.strip())
print(json.dumps(clean_json, indent=2))

--- 生のレスポンス ---
'\n{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["running"]\n  }\n}\n'

--- パース済みJSON ---
{
  "source": [
    "aws.ec2"
  ],
  "detail-type": [
    "EC2 Instance State-change Notification"
  ],
  "detail": {
    "state": [
      "running"
    ]
  }
}


---

# STEP 7: プロンプト評価ワークフロー

プロンプトを「なんとなくテスト」するのではなく、スコアで客観的に測定・比較する仕組みを作ります。

**5ステップの流れ：**
1. プロンプトを作成する
2. 評価データセットを用意する
3. 各質問をClaudeに送って回答を得る
4. 採点者（Claude）にスコアをつけさせる
5. プロンプトを改善して繰り返す

## 19. STEP1〜3: プロンプトと評価データセットを用意し、Claudeに回答させる

In [19]:
# STEP1: 評価対象のプロンプト（シンプルな初期バージョン）
def build_prompt(question):
    return f"""ユーザーの質問に答えてください:

{question}"""

# STEP2: 評価データセット
questions = [
    "2+2はいくつですか？",
    "オートミールの作り方を教えてください",
    "月までの距離はどれくらいですか？",
]

# STEP3: 各質問をClaudeに送って回答を収集
answers = []
for question in questions:
    messages = []
    add_user_message(messages, build_prompt(question))
    answer = chat(messages, temperature=0.0)  # 再現性のため低温に設定
    answers.append(answer)
    print(f"Q: {question}")
    print(f"A: {answer[:100]}...")  # 長い場合は先頭100文字だけ表示
    print()

Q: 2+2はいくつですか？
A: 2+2は**4**です。...

Q: オートミールの作り方を教えてください
A: # オートミールの基本的な作り方

## 基本レシピ（1人分）

### 材料
- オートミール：30-50g
- 水または牛乳：150-200ml
- 塩：ひとつまみ（お好みで）

## 調理方法
...

Q: 月までの距離はどれくらいですか？
A: # 月までの距離

月までの距離は**約38万4,400km**です。

## 詳細

- **平均距離**: 約384,400km（238,855マイル）
- **最接近時（近地点）**: 約356...



## 20. STEP4: 採点者（Claude）にスコアをつけさせる

In [20]:
def grade_answer(question, answer):
    """Claudeに回答を1〜10で採点させる"""
    grader_prompt = f"""あなたは客観的な採点者です。以下の回答を1〜10で採点してください。
10 = 完璧な回答、1 = 完全に間違いまたは役に立たない回答。
1から10の整数のみを返してください。それ以外のテキストは含めないでください。

質問: {question}
回答: {answer}"""

    messages = []
    add_user_message(messages, grader_prompt)
    score_text = chat(messages, temperature=0.0)
    return int(score_text.strip())

# 各回答をスコアリング
scores = []
for question, answer in zip(questions, answers):
    score = grade_answer(question, answer)
    scores.append(score)
    print(f"Q: {question}")
    print(f"スコア: {score}/10")
    print()

avg_score = sum(scores) / len(scores)
print(f"平均スコア: {avg_score:.2f}/10")

Q: 2+2はいくつですか？
スコア: 10/10

Q: オートミールの作り方を教えてください
スコア: 9/10

Q: 月までの距離はどれくらいですか？
スコア: 10/10

平均スコア: 9.67/10


## 21. STEP5: プロンプトを改善して再評価する

プロンプトに「詳しく答えてください」を追加して、スコアが変わるか比較します。

In [21]:
# 改善版プロンプト
def build_prompt_v2(question):
    return f"""ユーザーの質問に答えてください:

{question}

十分な詳細を含めて答えてください。"""

# 改善版で回答を収集してスコアリング
scores_v2 = []
for question in questions:
    messages = []
    add_user_message(messages, build_prompt_v2(question))
    answer = chat(messages, temperature=0.0)
    score = grade_answer(question, answer)
    scores_v2.append(score)
    print(f"Q: {question}")
    print(f"スコア: {score}/10")
    print()

avg_score_v2 = sum(scores_v2) / len(scores_v2)

print(f"初期プロンプト 平均スコア: {avg_score:.2f}/10")
print(f"改善プロンプト 平均スコア: {avg_score_v2:.2f}/10")
print(f"差分: {avg_score_v2 - avg_score:+.2f}")

Q: 2+2はいくつですか？
スコア: 10/10

Q: オートミールの作り方を教えてください
スコア: 9/10

Q: 月までの距離はどれくらいですか？
スコア: 9/10

初期プロンプト 平均スコア: 9.67/10
改善プロンプト 平均スコア: 9.33/10
差分: -0.33
